# Intro til Machine Learning — 6: Ekstra opgaver 😈

Velkommen til slutspillet. Disse opgaver er **større, friere og sværere** end dem i
notebook 1-5, og de kombinerer alt, hvad I har lært. Der er ingen fast rækkefølge — vælg
den, der frister mest. Hver opgave har startkode, så I aldrig starter fra en blank side.

| Opgave | Hvad | Bruger |
|---|---|---|
| E.1 | Pokédex-rapporten | pandas + plots |
| E.2 | Gradient descent i 2D | autograd + GD |
| E.3 | Type-oraklet (18 klasser!) | hele ML-pipelinen |
| E.4 | Den store aktiveringsdyst | MNIST + aktiveringer |
| E.5 ⭐ | Hyperparameter-jagten | pandas MØDER PyTorch |
| E.6 | Tegn selv et tal | MNIST + kreativitet |
| E.7 | Fejl-detektiven (boss-level 🐛×5) | ALT |
| E.8 ⭐ | Frit valg fra Kaggle | mod og eventyrlyst |

## Setup (kør altid denne først)

In [ ]:
!pip install -q kagglehub

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/hjaelpefunktioner.py

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from hjaelpefunktioner import vis_mnist_billeder

torch.manual_seed(42)
np.random.seed(42)

# Pokémon
sti = kagglehub.dataset_download("abcsds/pokemon")
df = pd.read_csv(sti + "/Pokemon.csv")

# MNIST (bruges i E.4, E.6 og E.7) — skåret ned som i notebook 5
sti_mnist = kagglehub.dataset_download("oddrationale/mnist-in-csv")
traen_df = pd.read_csv(sti_mnist + "/mnist_train.csv").sample(n=8000, random_state=42).reset_index(drop=True)
test_df = pd.read_csv(sti_mnist + "/mnist_test.csv").sample(n=2000, random_state=42).reset_index(drop=True)

X_mnist = torch.tensor(traen_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist = torch.tensor(traen_df["label"].values, dtype=torch.long)
X_mnist_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist_test = torch.tensor(test_df["label"].values, dtype=torch.long)

# 🚨 Plan B hvis Kaggle driller — fjern #'erne:
# df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/Pokemon.csv")
# traen_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_traen_lille.csv.gz")
# test_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_test_lille.csv.gz")

print("klar!", df.shape, X_mnist.shape)

### Opgave E.1 — Pokédex-rapporten 📊

I er blevet hyret som dataanalytikere af Professor Oak. Han vil vide: **hvilken generation
er den "bedste"?** Men "bedst" er jeres valg at definere — stærkest i snit? flest
legendariske? bedst balance mellem angreb og forsvar? mest variation?

**Krav til rapporten (byg den i denne notebook):**
1. Mindst **3 forskellige plots** (fx histogram, scatter, søjler)
2. Mindst én **groupby**
3. Mindst én **ny kolonne**, I selv har beregnet
4. En **konklusion i en markdown-celle**: hvilken generation vinder, og hvorfor?

Startkoden giver et første fingerpeg — resten er op til jer.

In [ ]:
# Første fingerpeg: gennemsnitlig Total pr. generation
df.groupby("Generation")["Total"].mean().plot(kind="bar")
plt.ylabel("gennemsnitlig Total")
plt.title("Er nyere generationer stærkere?")
plt.show()

# jeres analyse fortsætter her...

### Opgave E.2 — Gradient descent i 2D 🗺️

I notebook 2 rullede I ned ad en 1D-bakke. Rigtige tabslandskaber har millioner af
dimensioner — men i 2D kan vi stadig TEGNE det. Funktionen
$f(a, b) = (a-1)^2 + 3(b+2)^2$ er en oval dal med bund i $(1, -2)$.

**Jeres mission:** udfyld gradienten og opdateringsskridtene, og tegn ruten ned i dalen
oven på konturplottet. Prøv derefter forskellige startpunkter og læringsrater. (Bonus:
brug autograd i stedet for hånd-gradienten og tjek, at ruten er den samme.)

In [ ]:
def f(a, b):
    return (a - 1) ** 2 + 3 * (b + 2) ** 2

# Konturplottet ("landkortet" — hver ring er en højdekurve):
A, B = np.meshgrid(np.linspace(-4, 6, 100), np.linspace(-6, 2, 100))
plt.contour(A, B, f(A, B), levels=30, cmap="viridis")
plt.colorbar(label="f(a, b)")

# Gradient descent — udfyld de to partielle afledede og opdateringen:
a, b = 5.0, 1.0                  # startpunkt
laeringsrate = 0.1
rute_a, rute_b = [a], [b]
for i in range(40):
    grad_a = ...                 # ∂f/∂a = 2(a - 1)
    grad_b = ...                 # ∂f/∂b = 6(b + 2)
    a = ...
    b = ...
    rute_a.append(a)
    rute_b.append(b)

plt.plot(rute_a, rute_b, "o-", color="crimson", markersize=4)
plt.scatter([1], [-2], color="gold", s=150, zorder=3, edgecolors="black", label="minimum")
plt.legend()
plt.title(f"GD ender i ({a:.2f}, {b:.2f})")
plt.show()

### Opgave E.3 — Type-oraklet 🔮

Kan man se på en Pokémons stats, hvilken **type** den er? Det er klassifikation med
**18 klasser** — jeres hidtil vildeste. Udfyld hullerne (mønstret er 12.6 + notebook 5).

Bagefter: sammenlign med den dovne baseline (gæt altid på den mest almindelige type —
hvor god er DEN? Husk opgave 2.5!), og diskutér: hvorfor er det her SÅ meget sværere end
at spotte legendariske?

In [ ]:
typer = sorted(df["Type 1"].unique())
print(len(typer), "typer:", typer)
type_til_nr = {t: nr for nr, t in enumerate(typer)}

seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)
y_np = df["Type 1"].map(type_til_nr).values

X_traen_np, X_test_np, y_traen_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_traen, X_test = torch.tensor(X_traen_np), torch.tensor(X_test_np)
y_traen = torch.tensor(y_traen_np, dtype=...)        # ← hvilken dtype vil CrossEntropy have?
y_test = torch.tensor(y_test_np, dtype=torch.long)

class TypeOrakel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 64)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(64, ...)               # ← hvor mange klasser?

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

model = TypeOrakel()
tabsfunktion = ...                                    # ← tabsfunktionen?
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
for epoke in range(2000):
    optimizer.zero_grad()
    tab = tabsfunktion(model(X_traen), y_traen)
    tab.backward()
    optimizer.step()

with torch.no_grad():
    gaet = model(X_test).argmax(dim=1)
print(f"oraklets accuracy:  {(gaet == y_test).float().mean().item():.1%}")

# den dovne baseline: gæt ALTID på den mest almindelige type
...

### Opgave E.4 — Den store aktiveringsdyst 🥊

Afgør det én gang for alle (i hvert fald for MNIST): træn det SAMME netværk med **ReLU,
Sigmoid, Tanh og LeakyReLU** og vis de fire test-accuracies i ét søjlediagram. Skabelonen
har allerede løkken — I skal udfylde trænings-rytmen og accuracy-målingen (mønstret fra
notebook 5).

In [ ]:
resultater = {}

for navn, akt in [("ReLU", nn.ReLU()), ("Sigmoid", nn.Sigmoid()),
                  ("Tanh", nn.Tanh()), ("LeakyReLU", nn.LeakyReLU())]:
    torch.manual_seed(42)                       # fair kamp: samme startvægte
    model = nn.Sequential(nn.Linear(784, 128), akt, nn.Linear(128, 10))
    tabsfunktion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoke in range(3):
        for i in range(0, len(X_mnist), 64):
            ...                                  # ← trænings-rytmen (4 linjer)

    with torch.no_grad():
        acc = ...                                # ← test-accuracy (argmax!)
    resultater[navn] = acc
    print(f"{navn}: {acc:.1%}")

plt.bar(resultater.keys(), resultater.values())
plt.ylim(0.8, 1.0)
plt.ylabel("test-accuracy")
plt.title("Aktiveringsdysten (3 epoker)")
plt.show()

### Opgave E.5 ⭐ — Hyperparameter-jagten 🎯

Læringsrate og netværksstørrelse kaldes **hyperparametre** — tal, VI vælger, og som
modellen ikke selv lærer. I praksis prøver man sig systematisk frem. Nu skal pandas møde
PyTorch: kør en **dobbelt for-løkke** over læringsrater × skjulte størrelser (på et udsnit
af MNIST, så det går hurtigt), gem alle resultater i en **DataFrame**, og vis dem som et
**heatmap**. Udfyld hullerne — og find den bedste kombination!

In [ ]:
X_lille = X_mnist[:3000]
y_lille = y_mnist[:3000]

resultater = []
for lr in [0.0001, 0.001, 0.01]:
    for skjulte in [16, 64, 256]:
        torch.manual_seed(42)
        model = nn.Sequential(nn.Linear(784, skjulte), nn.ReLU(), nn.Linear(skjulte, 10))
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        tabsfunktion = nn.CrossEntropyLoss()
        for epoke in range(2):
            for i in range(0, len(X_lille), 64):
                ...                              # ← rytmen igen (4 linjer)

        with torch.no_grad():
            acc = (model(X_mnist_test).argmax(dim=1) == y_mnist_test).float().mean().item()
        resultater.append({"lr": lr, "skjulte": skjulte, "accuracy": acc})
        print(f"lr={lr}, skjulte={skjulte}: {acc:.1%}")

res_df = pd.DataFrame(resultater)
tabel = res_df.pivot(index="lr", columns="skjulte", values="accuracy")
print(tabel)

# heatmap af tabellen (imshow-mønstret fra notebook 1):
plt.imshow(tabel, cmap="viridis")
plt.colorbar(label="accuracy")
plt.xticks(range(len(tabel.columns)), tabel.columns)
plt.yticks(range(len(tabel.index)), tabel.index)
plt.xlabel("skjulte neuroner")
plt.ylabel("læringsrate")
plt.title("Hyperparameter-jagten")
plt.show()

### Opgave E.6 — Tegn selv et tal ✍️

Kan jeres netværk læse JERES håndskrift? Et 28×28-billede er bare et numpy-array, så I kan
"tegne" med slicing: `billede[raekker, kolonner] = 1.0` maler et hvidt felt. Startkoden
tegner et (meget kantet) 7-tal og spørger modellen.

**Jeres mission:** tegn mindst to andre cifre med slicing, og se om modellen kan læse dem.
Kig på sandsynlighederne — hvornår er den i tvivl? (Og hvis den fejler: er det jeres
håndskrift eller modellens skyld? 😄)

In [ ]:
# først: et hurtigtrænet netværk (samme opskrift som notebook 5)
torch.manual_seed(42)
model_e6 = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
optimizer = torch.optim.Adam(model_e6.parameters(), lr=0.001)
tabsfunktion = nn.CrossEntropyLoss()
for epoke in range(5):
    for i in range(0, len(X_mnist), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(model_e6(X_mnist[i:i + 64]), y_mnist[i:i + 64])
        tab.backward()
        optimizer.step()
print("model klar!")

# tegn et 7-tal med slicing — RET I DET HER og tegn jeres egne cifre:
billede = np.zeros((28, 28), dtype=np.float32)
billede[5:8, 6:21] = 1.0       # vandret streg foroven
billede[8:23, 17:20] = 1.0     # lodret streg ned

plt.imshow(billede, cmap="gray")
plt.show()

# hvad siger modellen?
with torch.no_grad():
    point = model_e6(torch.tensor(billede.reshape(1, 784)))
sandsynligheder = torch.softmax(point, dim=1).squeeze()
print("modellens gæt:", point.argmax().item())
plt.bar(range(10), sandsynligheder)
plt.xticks(range(10))
plt.title("modellens sandsynligheder")
plt.show()

### Opgave E.7 — Fejl-detektiven 🕵️ (boss-level)

Pipeline-koden nedenfor skal træne en legendarisk-spotter (som notebook 3) — men der
gemmer sig **FEM fejl** fra hele emnet i den. Nogle crasher med en fejlbesked, andre
snyder i **total stilhed**. Find og ret alle fem!

*Tip: I har mødt hver eneste af fejlene før. Tjek: dataforberedelsen, modellens output,
loop-rytmen (×2) — og til sidst: hvad der egentlig bliver målt.*

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
y_np = df["Legendary"].astype(int).values.astype("float32")

X_traen_np, X_test_np, y_traen_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_traen, X_test = torch.tensor(X_traen_np), torch.tensor(X_test_np)
y_traen, y_test = torch.tensor(y_traen_np), torch.tensor(y_test_np)

class Spotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(6, 16)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(16, 1)

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

model = Spotter()
tabsfunktion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoke in range(500):
    y_hat = model(X_traen).squeeze()
    tab = tabsfunktion(y_hat, y_traen)
    optimizer.step()
    tab.backward()

with torch.no_grad():
    gaet = (model(X_traen).squeeze() > 0.5).float()
accuracy = (gaet == y_traen).float().mean()
print(f"accuracy: {accuracy.item():.1%}")

### Opgave E.8 ⭐ — Frit valg fra Kaggle 🌍

Den sidste boss er friheden selv: **find jeres eget datasæt på
[kaggle.com/datasets](https://www.kaggle.com/datasets)** og træn et netværk på det.

**Tjekliste (= alt hvad I har lært):**
1. 🔍 Find et datasæt med en CSV-fil og mindst én kolonne, der er værd at forudsige.
   Gode begynder-emner: vin-kvalitet, bilpriser, fisk-vægt, film-ratings...
2. 📥 Hent det: `kagglehub.dataset_download("ejer/datasæt-navn")` (navnet står i datasættets URL).
3. 🧹 Udforsk og rens: `head`, `describe`, `isna().sum()` — drop eller udfyld huller,
   og vælg talkolonner som features.
4. 📏 Standardisér features. Train/test-split.
5. 🤔 Regression eller klassifikation? → vælg output-lag og tabsfunktion (opgave 11.8!).
6. 🏋️ Byg model, træn med rytmen, plot tabskurven.
7. 📊 Mål på testsættet — og sammenlign ALTID med en doven baseline (gennemsnittet /
   den største klasse).

Skabelonen nedenfor er jeres startgerüst — udfyld den trin for trin.

In [ ]:
# 1-2: hent jeres datasæt
sti_eget = kagglehub.dataset_download("...")     # ← udfyld
import os
print(os.listdir(sti_eget))                       # hvilke filer kom der?

eget_df = pd.read_csv(sti_eget + "/....csv")      # ← udfyld filnavnet
eget_df.head()

# Det var det hele! 🏁

Hvis I er nået hertil og har løst bare halvdelen: imponerende. I behersker nu hele kæden
fra rå CSV-fil til trænet neuralt netværk — og vigtigere: I ved, hvor fejlene gemmer sig,
og hvornår et flot tal lyver.

Vi ses til næste emne, hvor netværkene lærer at SE. 👁️🧠